# Comparing individual raw XSTAR runs

This notebook compares **two or more individual XSTAR run directories**. It is designed for raw XSTAR products (for example, `xout_spect1.fits`, `xout_cont1.fits`, `xout_lines1.fits`, and `xout_abund1.fits`), rather than stacked simulation grids.

For every run, the notebook:
- discovers available files and FITS extensions;
- reads the `PARAMETERS` extension when available;
- plots incident, transmitted, inward-emission, and outward-emission spectra;
- overlays the corresponding continuum using a dashed line in the same run colour;
- calculates outward emission minus continuum and transmitted/incident;
- compares runs on their native energy grids, and interpolates only when calculating residuals;
- provides access to line lists, RRCs, abundance/column/heating/cooling tables, and log files.

**Important:** edit the `RUNS` dictionary in the next cell. Each `path` must point to the directory containing the raw products of one XSTAR run.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
import astropy.units as u
import ipywidgets as widgets
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 100)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "axes.grid": False,
    "grid.alpha": 0.25,
    "font.size": 10,
})

## 1. Configure the XSTAR runs

In [ ]:
# Add two or more individual XSTAR run directories.
# Each path should contain the raw products of ONE XSTAR run.
RUNS = {
    "run_1": {
        "label": "Run 1",
        "path": Path("/data-fast/dragon/vicente/iongrid/runs/run_20261106_htc_lhs/points/p000000001"),
    },
    "run_2": {
        "label": "Run 2",
        "path": Path("/data-fast/dragon/vicente/iongrid/runs/run_20261106_htc_lhs/points/p000000027"),
    },
    # "run_3": {
    #     "label": "Run 3",
    #     "path": Path("/path/to/third/xstar/run"),
    # },
}

# Search inside subdirectories as well as directly inside each run directory.
SEARCH_SUBDIRECTORIES = False

# Energy units used for plotting. Standard xout_spect1/xout_cont1 energies
# are assumed to be eV if the FITS column does not specify a unit.
PLOT_ENERGY_UNIT = "keV"
ENERGY_LOW_LIM = 0.1
ENERGY_UP_LIM = 12.0

# Residual reference: None means the first configured run.
REFERENCE_RUN = None

# Residuals are calculated on the reference run's energy grid.
# Other runs are interpolated only for residual calculations, never for
# the native-grid spectrum overlays.
RESIDUAL_MODE = "difference"  # "difference" or "fractional"
SAVE_FIGURES = False
OUTPUT_DIR = Path("./xstar_raw_run_comparisons")

TABLE_PREVIEW_ROWS = 20
LOG_PREVIEW_CHARS = None  # None = show complete log files

if len(RUNS) < 2:
    raise ValueError("Configure at least two individual XSTAR runs.")

for run_id, cfg in RUNS.items():
    cfg["path"] = Path(cfg["path"]).expanduser()
    if not cfg["path"].is_dir():
        raise FileNotFoundError(
            f"{run_id} ({cfg['label']}): directory does not exist: {cfg['path']}"
        )

run_ids = list(RUNS)
if REFERENCE_RUN is None:
    reference_run_id = run_ids[0]
else:
    if REFERENCE_RUN not in RUNS:
        raise KeyError(f"REFERENCE_RUN={REFERENCE_RUN!r} is not configured.")
    reference_run_id = REFERENCE_RUN

print(f"Configured {len(RUNS)} runs: {[RUNS[k]['label'] for k in run_ids]}")
print(f"Residual reference: {RUNS[reference_run_id]['label']}")

## 2. Discover files and inspect FITS products

In [ ]:
EXPECTED_PRODUCTS = {
    "xout_spect1.fits": "Spectrum including line emission binned into the continuum",
    "xout_cont1.fits": "Continuum-only spectrum",
    "xout_lines1.fits": "Strong emission-line list",
    "xout_abund1.fits": "Zone-by-zone abundances, ionic columns, heating, and cooling",
    "xout_rrc1.fits": "Radiative recombination continuum list",
    "xout_step.log": "XSTAR run log and diagnostics",
}

def product_description(filename):
    name = filename.lower()
    if name in EXPECTED_PRODUCTS:
        return EXPECTED_PRODUCTS[name]
    if re.match(r"^xo\d{2}_(?:detail|detal)\d+\.fits$", name):
        return "Optional detailed XSTAR output"
    if name.endswith((".fits", ".fit", ".fts")):
        return "Additional FITS product"
    if name.endswith((".log", ".out")):
        return "Additional text/log output"
    return "Other run output"

def discover_run_files(cfg):
    root = cfg["path"]
    iterator = root.rglob("*") if SEARCH_SUBDIRECTORIES else root.iterdir()
    paths = sorted(
        (p for p in iterator if p.is_file()),
        key=lambda p: str(p).lower(),
    )
    fits_paths = [p for p in paths if p.suffix.lower() in {".fits", ".fit", ".fts"}]
    log_paths = [p for p in paths if p.suffix.lower() in {".log", ".out"}]
    return paths, fits_paths, log_paths

RUN_FILES = {}
inventory_frames = []

for run_id, cfg in RUNS.items():
    paths, fits_paths, log_paths = discover_run_files(cfg)
    RUN_FILES[run_id] = {
        "all": paths,
        "fits": fits_paths,
        "logs": log_paths,
    }
    for p in paths:
        inventory_frames.append({
            "run": cfg["label"],
            "file": str(p.relative_to(cfg["path"])),
            "type": p.suffix.lower() or "(no extension)",
            "size [MiB]": round(p.stat().st_size / 1024**2, 3),
            "description": product_description(p.name),
        })

file_inventory = pd.DataFrame(inventory_frames)
display(file_inventory if not file_inventory.empty else
        pd.DataFrame({"message": ["No files found."]}))

def inventory_fits_file(run_id, filepath):
    rows = []
    with fits.open(filepath, memmap=False, lazy_load_hdus=True) as hdul:
        for hdu_index, hdu in enumerate(hdul):
            is_table = isinstance(hdu, (fits.BinTableHDU, fits.TableHDU))
            columns = list(hdu.columns.names or []) if is_table else []
            rows.append({
                "run": RUNS[run_id]["label"],
                "file": str(filepath.relative_to(RUNS[run_id]["path"])),
                "HDU": hdu_index,
                "class": type(hdu).__name__,
                "EXTNAME": hdu.header.get("EXTNAME", "(primary / unnamed)"),
                "rows": hdu.header.get("NAXIS2") if is_table else None,
                "columns": ", ".join(columns),
            })
    return rows

hdu_rows = []
for run_id in run_ids:
    for filepath in RUN_FILES[run_id]["fits"]:
        try:
            hdu_rows.extend(inventory_fits_file(run_id, filepath))
        except Exception as exc:
            hdu_rows.append({
                "run": RUNS[run_id]["label"],
                "file": filepath.name,
                "HDU": None,
                "class": "READ ERROR",
                "EXTNAME": "",
                "rows": None,
                "columns": str(exc),
            })

hdu_inventory = pd.DataFrame(hdu_rows)
display(hdu_inventory if not hdu_inventory.empty else
        pd.DataFrame({"message": ["No FITS HDUs to inspect."]}))

## 3. FITS helpers and load raw spectral products

In [ ]:
def native_array(values):
    arr = np.asarray(values)
    if arr.dtype.kind in "biufc" and arr.dtype.byteorder == ">":
        arr = arr.astype(arr.dtype.newbyteorder("="), copy=True)
    return arr

def normalise_name(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())

def matching_column(columns, candidates):
    normalised = {normalise_name(c): c for c in columns}
    for candidate in candidates:
        key = normalise_name(candidate)
        if key in normalised:
            return normalised[key]
    return None

def find_hdu_index(filepath, extname=None, required_columns=None):
    required = [str(x).lower() for x in (required_columns or [])]
    with fits.open(filepath, memmap=False, lazy_load_hdus=True) as hdul:
        for i, hdu in enumerate(hdul):
            if not isinstance(hdu, (fits.BinTableHDU, fits.TableHDU)):
                continue
            name = str(hdu.header.get("EXTNAME", "")).strip().lower()
            if extname is not None and name != extname.lower():
                continue
            names = [str(x).lower() for x in (hdu.columns.names or [])]
            if all(col in names for col in required):
                return i
    return None

def read_table(filepath, hdu_index):
    with fits.open(filepath, memmap=False) as hdul:
        hdu = hdul[hdu_index]
        if not isinstance(hdu, (fits.BinTableHDU, fits.TableHDU)) or hdu.data is None:
            raise TypeError(f"HDU {hdu_index} in {filepath.name} is not a populated table.")
        data = {}
        for name in hdu.columns.names:
            arr = native_array(hdu.data[name])
            data[name] = arr if arr.ndim <= 1 else list(arr)
        frame = pd.DataFrame(data)
        units = {col.name: (col.unit or "") for col in hdu.columns}
        header = hdu.header.copy()
    return frame, units, header

def energy_to_plot_unit(values, unit, filepath):
    values = np.asarray(values, dtype=float).reshape(-1)
    source_unit = unit or (
        "eV" if filepath.name.lower() in {"xout_spect1.fits", "xout_cont1.fits"} else None
    )
    if source_unit is None:
        warnings.warn(
            f"{filepath.name}: no energy unit specified; values are left unchanged."
        )
        return values
    try:
        return (values * u.Unit(source_unit)).to_value(u.Unit(PLOT_ENERGY_UNIT))
    except Exception as exc:
        warnings.warn(f"Could not convert {source_unit!r} to {PLOT_ENERGY_UNIT}: {exc}")
        return values

def locate_product(run_id, filename):
    matches = [
        p for p in RUN_FILES[run_id]["all"]
        if p.name.lower() == filename.lower()
    ]
    return matches[0] if matches else None

def load_spectral_product(run_id, filename):
    filepath = locate_product(run_id, filename)
    if filepath is None:
        return None

    idx = find_hdu_index(filepath, extname="XSTAR_SPECTRA", required_columns=["energy"])
    if idx is None:
        idx = find_hdu_index(filepath, required_columns=["energy"])
    if idx is None:
        warnings.warn(f"No table with an energy column found in {filepath}.")
        return None

    frame, units, header = read_table(filepath, idx)
    energy_col = matching_column(frame.columns, ["energy"])
    energy = np.asarray(frame[energy_col], dtype=float).squeeze()
    if energy.ndim != 1:
        warnings.warn(f"Energy column in {filepath.name} is not one-dimensional.")
        return None
    energy = energy_to_plot_unit(energy, units.get(energy_col), filepath)

    components = {}
    definitions = {
        "incident": ["incident"],
        "transmitted": ["transmitted"],
        "emit_inward": ["emit_inward", "emitinward"],
        "emit_outward": ["emit_outward", "emitoutward"],
    }
    for key, candidates in definitions.items():
        col = matching_column(frame.columns, candidates)
        if col is not None:
            values = np.asarray(frame[col], dtype=float).squeeze()
            if values.ndim == 1 and values.size == energy.size:
                components[key] = values
            else:
                warnings.warn(f"{filepath.name}: skipping {col}; incompatible shape.")
    return {
        "path": filepath,
        "energy": energy,
        "components": components,
        "frame": frame,
        "units": units,
        "header": header,
    }

RAW = {}
for run_id in run_ids:
    RAW[run_id] = {
        "spectrum": load_spectral_product(run_id, "xout_spect1.fits"),
        "continuum": load_spectral_product(run_id, "xout_cont1.fits"),
    }
    if RAW[run_id]["spectrum"] is None:
        warnings.warn(f"{RUNS[run_id]['label']}: xout_spect1.fits was not found/readable.")
    if RAW[run_id]["continuum"] is None:
        warnings.warn(f"{RUNS[run_id]['label']}: xout_cont1.fits was not found/readable.")

print("Loaded raw spectral products.")

## 4. Read and compare run parameters

In [ ]:
def load_parameters(run_id):
    candidates = RUN_FILES[run_id]["fits"]
    for filepath in candidates:
        idx = find_hdu_index(filepath, extname="PARAMETERS")
        if idx is not None:
            frame, units, header = read_table(filepath, idx)
            return {"frame": frame, "units": units, "file": filepath, "header": header}
    return None

PARAMETERS = {run_id: load_parameters(run_id) for run_id in run_ids}

parameter_rows = []
for run_id in run_ids:
    result = PARAMETERS[run_id]
    if result is None:
        parameter_rows.append({
            "run": RUNS[run_id]["label"],
            "parameter": "(PARAMETERS extension not found)",
            "value": "",
            "unit": "",
        })
        continue
    frame = result["frame"]
    if len(frame) == 0:
        continue
    row = frame.iloc[0]
    for col in frame.columns:
        value = row[col]
        parameter_rows.append({
            "run": RUNS[run_id]["label"],
            "parameter": col,
            "value": value,
            "unit": result["units"].get(col, ""),
        })

parameter_comparison = pd.DataFrame(parameter_rows)
display(parameter_comparison)

# Compact summary of loaded spectral products and energy grids.
summary_rows = []
for run_id in run_ids:
    for kind in ("spectrum", "continuum"):
        product = RAW[run_id][kind]
        if product is None:
            continue
        energy = product["energy"]
        summary_rows.append({
            "run": RUNS[run_id]["label"],
            "product": kind,
            "energy bins": len(energy),
            f"minimum [{PLOT_ENERGY_UNIT}]": np.nanmin(energy),
            f"maximum [{PLOT_ENERGY_UNIT}]": np.nanmax(energy),
            "components": ", ".join(product["components"]),
        })
display(pd.DataFrame(summary_rows))

## 5. Compare the energy grids

Each run is plotted against its **own native energy grid**. Different bin counts or energy grids are allowed.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
RUN_COLORS = {run_id: colors[i % len(colors)] for i, run_id in enumerate(run_ids)}

for run_id in run_ids:
    product = RAW[run_id]["spectrum"]
    if product is None:
        continue
    energy = product["energy"]
    ax.plot(
        np.arange(len(energy)),
        energy,
        color=RUN_COLORS[run_id],
        linewidth=1.3,
        label=RUNS[run_id]["label"],
    )

ax.set_yscale("log")
ax.set_xlabel("Energy bin / channel index")
ax.set_ylabel(f"Energy ({PLOT_ENERGY_UNIT})")
ax.set_title("Native energy grids of the raw XSTAR runs")
ax.legend()
fig.tight_layout()
plt.show()

for run_id in run_ids:
    spec = RAW[run_id]["spectrum"]
    cont = RAW[run_id]["continuum"]
    if spec is None or cont is None:
        continue
    same = (
        spec["energy"].shape == cont["energy"].shape
        and np.allclose(spec["energy"], cont["energy"], rtol=1e-10, atol=1e-12, equal_nan=True)
    )
    print(f"{RUNS[run_id]['label']}: spectrum/continuum energy grids {'match' if same else 'DIFFER'}")

## 6. Compare the six spectral quantities

The first four panels show each run's spectrum as a solid line and its continuum as a dashed line, using the **same colour for the same run**.

The final two panels show:
- outward emission spectrum minus outward continuum;
- transmitted divided by incident, using `xout_spect1.fits`.

The original spectra are always plotted on their native grids. For the derived quantities, the continuum is interpolated onto the corresponding spectrum grid if necessary.

In [ ]:
COMPONENTS = [
    ("incident", "Incident"),
    ("transmitted", "Transmitted"),
    ("emit_inward", "Emission inward"),
    ("emit_outward", "Emission outward"),
]

def energy_mask(energy):
    return (
        np.isfinite(energy)
        & (energy >= ENERGY_LOW_LIM)
        & (energy <= ENERGY_UP_LIM)
    )

def interpolate_values(source_energy, source_values, target_energy):
    valid = (
        np.isfinite(source_energy)
        & np.isfinite(source_values)
        & (source_energy > 0)
    )
    if valid.sum() < 2:
        return np.full(target_energy.shape, np.nan)
    x = source_energy[valid]
    y = source_values[valid]
    order = np.argsort(x)
    x, y = x[order], y[order]
    # np.interp requires increasing x. Collapse duplicate energies if present.
    unique_x, unique_indices = np.unique(x, return_index=True)
    y = y[unique_indices]
    if unique_x.size < 2:
        return np.full(target_energy.shape, np.nan)
    return np.interp(target_energy, unique_x, y, left=np.nan, right=np.nan)

def plot_spectral_comparison():
    fig, axes = plt.subplots(3, 2, figsize=(15, 13))
    axes = axes.ravel()

    # Panels 1-4: raw spectrum and continuum, both on native grids.
    for panel_index, (component, title) in enumerate(COMPONENTS):
        ax = axes[panel_index]
        plotted = False

        for run_id in run_ids:
            spec_product = RAW[run_id]["spectrum"]
            cont_product = RAW[run_id]["continuum"]
            color = RUN_COLORS[run_id]
            label = RUNS[run_id]["label"]

            if spec_product is not None and component in spec_product["components"]:
                energy = spec_product["energy"]
                values = spec_product["components"][component]
                mask = energy_mask(energy) & np.isfinite(values)
                if mask.any():
                    ax.plot(
                        energy[mask], values[mask],
                        color=color, linestyle="-", linewidth=1.25,
                        label=f"{label} — spectrum",
                    )
                    plotted = True

            if cont_product is not None and component in cont_product["components"]:
                energy = cont_product["energy"]
                values = cont_product["components"][component]
                mask = energy_mask(energy) & np.isfinite(values)
                if mask.any():
                    ax.plot(
                        energy[mask], values[mask],
                        color=color, linestyle="--", linewidth=0.9,
                        label=f"{label} — continuum",
                    )
                    plotted = True

        ax.set_title(title)
        ax.set_xscale("log")
        ax.set_xlim(ENERGY_LOW_LIM, ENERGY_UP_LIM)
        ax.set_xlabel(f"Energy ({PLOT_ENERGY_UNIT})")
        ax.set_ylabel("Specific luminosity (see XSTAR units/manual)")
        if plotted:
            positive = [
                line.get_ydata()[
                    np.isfinite(line.get_ydata()) & (line.get_ydata() > 0)
                ]
                for line in ax.lines
            ]
            if any(x.size for x in positive) and all(
                np.all(x > 0) for x in positive if x.size
            ):
                ax.set_yscale("log")
            ax.legend(fontsize=7, ncol=1)
        else:
            ax.text(0.5, 0.5, "No compatible data", ha="center", va="center",
                    transform=ax.transAxes)

    # Panel 5: outward emission spectrum - continuum.
    ax = axes[4]
    for run_id in run_ids:
        spec_product = RAW[run_id]["spectrum"]
        cont_product = RAW[run_id]["continuum"]
        if (
            spec_product is None or cont_product is None
            or "emit_outward" not in spec_product["components"]
            or "emit_outward" not in cont_product["components"]
        ):
            continue

        energy = spec_product["energy"]
        spectrum = spec_product["components"]["emit_outward"]
        continuum_interp = interpolate_values(
            cont_product["energy"],
            cont_product["components"]["emit_outward"],
            energy,
        )
        difference = spectrum - continuum_interp
        mask = energy_mask(energy) & np.isfinite(difference)
        if mask.any():
            ax.plot(
                energy[mask], difference[mask],
                color=RUN_COLORS[run_id], linewidth=1.1,
                label=RUNS[run_id]["label"],
            )

    ax.axhline(0, color="black", linestyle=":", linewidth=0.8)
    ax.set_title("Emission outward: spectrum − continuum")
    ax.set_xscale("log")
    ax.set_xlim(ENERGY_LOW_LIM, ENERGY_UP_LIM)
    ax.set_xlabel(f"Energy ({PLOT_ENERGY_UNIT})")
    ax.set_ylabel("Difference in specific luminosity")
    if ax.lines:
        ax.legend(fontsize=8)

    # Panel 6: transmitted / incident from xout_spect1.fits.
    ax = axes[5]
    for run_id in run_ids:
        product = RAW[run_id]["spectrum"]
        if product is None:
            continue
        comps = product["components"]
        if "incident" not in comps or "transmitted" not in comps:
            continue

        energy = product["energy"]
        incident = comps["incident"]
        transmitted = comps["transmitted"]
        ratio = np.full(energy.shape, np.nan)
        valid = (
            np.isfinite(incident)
            & np.isfinite(transmitted)
            & (incident != 0)
        )
        ratio[valid] = transmitted[valid] / incident[valid]
        mask = energy_mask(energy) & np.isfinite(ratio)

        if mask.any():
            ax.plot(
                energy[mask], ratio[mask],
                color=RUN_COLORS[run_id], linewidth=1.1,
                label=RUNS[run_id]["label"],
            )

    ax.axhline(1, color="black", linestyle=":", linewidth=0.8, label="Ratio = 1")
    ax.set_title("Transmission: transmitted / incident")
    ax.set_xscale("log")
    ax.set_xlim(ENERGY_LOW_LIM, ENERGY_UP_LIM)
    ax.set_xlabel(f"Energy ({PLOT_ENERGY_UNIT})")
    ax.set_ylabel("Transmitted / Incident")
    if ax.lines:
        ax.legend(fontsize=8)

    fig.suptitle("Comparison of individual raw XSTAR runs", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    if SAVE_FIGURES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_DIR / "six_panel_spectral_comparison.png", bbox_inches="tight")
    plt.show()

plot_spectral_comparison()

## 7. Residual comparisons against a reference run

This section compares each run with the selected reference. The upper panel shows the native-grid spectra. The lower panel shows residuals evaluated on the **reference run's energy grid**; non-reference spectra are interpolated for this calculation only.

For `RESIDUAL_MODE = "difference"`, residual = run − reference.

For `RESIDUAL_MODE = "fractional"`, residual = (run − reference) / reference. Bins where the reference is zero or non-finite are omitted.

Use the dropdown to choose the component.

In [ ]:
def plot_component_residuals(component="emit_outward"):
    if component not in dict(COMPONENTS):
        raise ValueError(f"Unknown component: {component}")

    ref_product = RAW[reference_run_id]["spectrum"]
    if ref_product is None or component not in ref_product["components"]:
        raise ValueError("The reference run does not contain the selected component.")

    ref_energy = ref_product["energy"]
    ref_values = ref_product["components"][component]
    ref_mask = energy_mask(ref_energy)

    fig, (ax_top, ax_bottom) = plt.subplots(
        2, 1, figsize=(12, 7), sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05},
    )

    # Native-grid spectra.
    for run_id in run_ids:
        product = RAW[run_id]["spectrum"]
        if product is None or component not in product["components"]:
            continue
        energy = product["energy"]
        values = product["components"][component]
        mask = energy_mask(energy) & np.isfinite(values)
        if mask.any():
            ax_top.plot(
                energy[mask], values[mask],
                color=RUN_COLORS[run_id],
                linestyle="-" if run_id == reference_run_id else "--",
                linewidth=1.25,
                label=RUNS[run_id]["label"],
            )

    # Residuals on reference grid.
    for run_id in run_ids:
        if run_id == reference_run_id:
            continue
        product = RAW[run_id]["spectrum"]
        if product is None or component not in product["components"]:
            continue

        values_on_ref = interpolate_values(
            product["energy"],
            product["components"][component],
            ref_energy,
        )

        if RESIDUAL_MODE == "difference":
            residual = values_on_ref - ref_values
            ylabel = "Run − reference"
        elif RESIDUAL_MODE == "fractional":
            residual = np.full(ref_energy.shape, np.nan)
            valid_ref = np.isfinite(ref_values) & (ref_values != 0)
            residual[valid_ref] = (
                values_on_ref[valid_ref] - ref_values[valid_ref]
            ) / ref_values[valid_ref]
            ylabel = "(Run − reference) / reference"
        else:
            raise ValueError("RESIDUAL_MODE must be 'difference' or 'fractional'.")

        mask = ref_mask & np.isfinite(residual)
        if mask.any():
            ax_bottom.plot(
                ref_energy[mask], residual[mask],
                color=RUN_COLORS[run_id], linewidth=1.1,
                label=RUNS[run_id]["label"],
            )

    ax_top.set_title(
        f"{dict(COMPONENTS)[component]} — reference: {RUNS[reference_run_id]['label']}"
    )
    ax_top.set_ylabel("Specific luminosity (see XSTAR units/manual)")
    ax_top.set_xscale("log")
    ax_top.set_xlim(ENERGY_LOW_LIM, ENERGY_UP_LIM)
    ax_top.legend(fontsize=8)

    ax_bottom.axhline(0, color="black", linestyle=":", linewidth=0.8)
    ax_bottom.set_xlabel(f"Energy ({PLOT_ENERGY_UNIT})")
    ax_bottom.set_ylabel(ylabel)
    ax_bottom.set_xscale("log")
    ax_bottom.set_xlim(ENERGY_LOW_LIM, ENERGY_UP_LIM)
    ax_bottom.legend(fontsize=8)

    fig.tight_layout()
    if SAVE_FIGURES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_DIR / f"residual_{component}.png", bbox_inches="tight")
    plt.show()

component_widget = widgets.Dropdown(
    options=[(title, key) for key, title in COMPONENTS],
    value="emit_outward",
    description="Component:",
    layout=widgets.Layout(width="350px"),
)
display(widgets.interactive(
    plot_component_residuals,
    component=component_widget,
))

## 8. Compare line lists and radiative recombination continua

In [ ]:
def find_product_table(run_id, filename, preferred_extnames=()):
    filepath = locate_product(run_id, filename)
    if filepath is None:
        return None
    for extname in preferred_extnames:
        idx = find_hdu_index(filepath, extname=extname)
        if idx is not None:
            return read_table(filepath, idx)[0]
    with fits.open(filepath, memmap=False, lazy_load_hdus=True) as hdul:
        for idx, hdu in enumerate(hdul):
            if isinstance(hdu, (fits.BinTableHDU, fits.TableHDU)) and hdu.data is not None:
                return read_table(filepath, idx)[0]
    return None

def show_transition_tables(filename, preferred_extnames=()):
    for run_id in run_ids:
        frame = find_product_table(run_id, filename, preferred_extnames)
        print(f"\n{RUNS[run_id]['label']} — {filename}")
        if frame is None:
            print("Product not found or no readable table.")
        else:
            print(f"{len(frame):,} rows")
            display(frame.head(TABLE_PREVIEW_ROWS))

show_transition_tables("xout_lines1.fits", ("XSTAR_LINES",))
show_transition_tables("xout_rrc1.fits", ("XSTAR_SPECTRA",))

## 9. Explore abundance, column, heating, and cooling tables

In [ ]:
def show_abundance_tables(run_id):
    filepath = locate_product(run_id, "xout_abund1.fits")
    if filepath is None:
        print(f"{RUNS[run_id]['label']}: xout_abund1.fits not found.")
        return

    for extname in ("ABUNDANCES", "COLUMNS", "HEATING", "COOLING"):
        idx = find_hdu_index(filepath, extname=extname)
        print(f"\n{RUNS[run_id]['label']} — {extname}")
        if idx is None:
            print("Extension not found.")
            continue

        frame, units, _ = read_table(filepath, idx)
        display(frame.head(TABLE_PREVIEW_ROWS))

        numeric = []
        for col in frame.columns:
            arr = np.asarray(frame[col])
            if arr.ndim == 1 and arr.dtype.kind in "biufc":
                numeric.append(col)

        if len(numeric) < 2:
            continue

        x_default = matching_column(numeric, ["radius"]) or numeric[0]
        y_options = [c for c in numeric if c != x_default]
        y_default = matching_column(
            y_options, ["temperature", "ion_parameter", "x_e", "n_p"]
        ) or y_options[0]

        # Define a closure so only the four plotting choices become widgets.
        def make_plotter(frame, units, extname, run_label):
            def _plot(x_column=x_default, y_column=y_default,
                      x_scale="log", y_scale="log"):
                x = pd.to_numeric(frame[x_column], errors="coerce").to_numpy(dtype=float)
                y = pd.to_numeric(frame[y_column], errors="coerce").to_numpy(dtype=float)
                mask = np.isfinite(x) & np.isfinite(y)
                if x_scale == "log":
                    mask &= x > 0
                if y_scale == "log":
                    mask &= y > 0
                if not mask.any():
                    print("No finite points for the selected axis scales.")
                    return

                order = np.argsort(x[mask])
                fig, ax = plt.subplots(figsize=(9, 4.5))
                ax.plot(x[mask][order], y[mask][order], marker=".", ms=4, lw=1)
                ax.set_xscale(x_scale)
                ax.set_yscale(y_scale)
                ax.set_xlabel(f"{x_column} [{units.get(x_column, '')}]")
                ax.set_ylabel(f"{y_column} [{units.get(y_column, '')}]")
                ax.set_title(f"{run_label} — {extname}")
                fig.tight_layout()
                plt.show()
            return _plot

        plotter = make_plotter(frame, units, extname, RUNS[run_id]["label"])
        controls = widgets.interactive(
            plotter,
            x_column=widgets.Dropdown(options=numeric, value=x_default, description="X:"),
            y_column=widgets.Dropdown(options=y_options, value=y_default, description="Y:"),
            x_scale=widgets.ToggleButtons(options=["linear", "log"], value="log", description="X scale:"),
            y_scale=widgets.ToggleButtons(options=["linear", "log"], value="log", description="Y scale:"),
        )
        display(controls)

for run_id in run_ids:
    show_abundance_tables(run_id)

## 10. Display XSTAR run logs

In [ ]:
for run_id in run_ids:
    log_paths = RUN_FILES[run_id]["logs"]
    print("\n" + "=" * 100)
    print(RUNS[run_id]["label"])
    print("=" * 100)

    if not log_paths:
        print("No .log or .out files found.")
        continue

    for log_path in log_paths:
        print("\n" + "-" * 80)
        print(log_path.relative_to(RUNS[run_id]["path"]))
        print("-" * 80)
        try:
            content = log_path.read_text(errors="replace")
            if LOG_PREVIEW_CHARS is not None and len(content) > LOG_PREVIEW_CHARS:
                print(content[:LOG_PREVIEW_CHARS])
                print(
                    f"\n[Truncated after {LOG_PREVIEW_CHARS:,} characters; "
                    "set LOG_PREVIEW_CHARS=None to display the complete file.]"
                )
            else:
                print(content)
        except Exception as exc:
            print(f"Could not read {log_path.name}: {exc}")